# 02. Data Preprocessing & Leakage Prevention — Thyroid Disease Detection

**Project:** Thyroid Disease Detection Using Machine Learning  
**Author:** Venkatesh Devarakonda  
**Curriculum:** B.Tech CSE (AI & ML) — Semester 2-1 Academic Project  

---

## 1. Objective
This notebook demonstrates the complete preprocessing pipeline:
- Attribute selection (dropping administrative & non-informative columns).
- Categorical and binary feature encoding.
- Missing value imputation using median/mean.
- Outlier inspection using the Interquartile Range (IQR) method.
- Class balancing concepts (Random Over-sampling).
- Stratified Train-Test splitting (80% Train, 20% Test).
- Feature standardization using `StandardScaler` with strict prevention of data leakage.


In [1]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

# Ensure src module is importable
sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("."))

from src.data_preprocessing import (
    load_processed_dataset,
    split_data,
    scale_features,
    preprocess_pipeline,
    CLASS_MAP,
)
print("Modules loaded successfully!")


Modules loaded successfully!


## 2. Load Processed Dataset

In [2]:
proc_path = os.path.join("..", "data", "processed", "thyroid_processed.csv")
if not os.path.exists(proc_path):
    proc_path = os.path.join("data", "processed", "thyroid_processed.csv")

df = pd.read_csv(proc_path)
print("Dataset shape:", df.shape)
print("Features:", list(df.columns[:-1]))
print("Target:", df.columns[-1])


Dataset shape: (20000, 15)
Features: ['age', 'sex', 'on thyroxine', 'on antithyroid medication', 'pregnant', 'thyroid surgery', 'lithium', 'goitre', 'tumor', 'TSH', 'T3', 'TT4', 'T4U', 'FTI']
Target: class


## 3. Outlier Inspection via IQR Method
In clinical datasets, extreme laboratory readings often represent acute pathology rather than measurement errors.
Here we inspect feature ranges using the IQR capping approach tested in the original project:
$$IQR = Q_3 - Q_1$$
$$	ext{Lower Bound} = Q_1 - 1.5 	imes IQR, \quad 	ext{Upper Bound} = Q_3 + 1.5 	imes IQR$$


In [3]:
def inspect_iqr_bounds(dataframe, cols):
    iqr_summary = []
    for col in cols:
        q1 = dataframe[col].quantile(0.25)
        q3 = dataframe[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        outliers_count = ((dataframe[col] < lower) | (dataframe[col] > upper)).sum()
        iqr_summary.append({
            "Feature": col,
            "Q1": round(q1, 2),
            "Q3": round(q3, 2),
            "IQR": round(iqr, 2),
            "Lower Bound": round(lower, 2),
            "Upper Bound": round(upper, 2),
            "Outlier Count": outliers_count,
            "Outlier %": round(outliers_count / len(dataframe) * 100, 2)
        })
    return pd.DataFrame(iqr_summary)

num_features = ["age", "TSH", "T3", "TT4", "T4U", "FTI"]
inspect_iqr_bounds(df, num_features)


,Feature,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier %
0,age,41.00,62.00,21.00,9.50,93.50,148,0.74
1,TSH,1.50,23.00,21.50,-30.75,55.25,2332,11.66
2,T3,1.20,2.20,1.00,-0.30,3.70,181,0.90
3,TT4,32.00,98.00,66.00,-67.00,197.00,86,0.43
4,T4U,0.89,1.07,0.18,0.62,1.34,736,3.68
5,FTI,54.00,110.26,56.26,-30.39,194.66,93,0.46


## 4. Stratified Train-Test Split (80 / 20)
We separate features $X$ and target $y$, then apply stratified sampling so that training and testing partitions retain identical class ratios:


In [4]:
X_train, X_test, y_train, y_test = split_data(
    df, target_column="class", test_size=0.20, random_state=42
)

print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print("\nTraining Class Proportions:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTesting Class Proportions:")
print(y_test.value_counts(normalize=True).round(3))


X_train shape: (16000, 14) | y_train shape: (16000,)
X_test shape:  (4000, 14)  | y_test shape:  (4000,)

Training Class Proportions:
class
11    0.25
1     0.25
7     0.25
9     0.25
Name: proportion, dtype: float64

Testing Class Proportions:
class
11    0.25
1     0.25
9     0.25
7     0.25
Name: proportion, dtype: float64


## 5. Feature Scaling & Prevention of Data Leakage

### What is Data Leakage?
Data leakage occurs when information from outside the training dataset (such as test set statistics) is inadvertently used to fit preprocessing transformations.

### Proper Protocol:
1. **Fit** the scaler ONLY on `X_train`: $\mu_{train}, \sigma_{train}$.
2. **Transform** `X_train` using $\mu_{train}, \sigma_{train}$.
3. **Transform** `X_test` using the SAME $\mu_{train}, \sigma_{train}$ (never calling `fit` or `fit_transform` on test!).


In [5]:
scaler_path = os.path.join("..", "models", "scaler.pkl")
if not os.path.exists(os.path.dirname(scaler_path)):
    scaler_path = os.path.join("models", "scaler.pkl")

X_train_scaled, X_test_scaled, scaler = scale_features(
    X_train, X_test, scaler_save_path=scaler_path
)

print("StandardScaler Mean vector (computed strictly from X_train):")
for f, m in zip(X_train.columns, scaler.mean_):
    print(f"  {f:25s}: {m:.3f}")

print(f"\nStandardScaler artifact saved to {scaler_path}")


StandardScaler Mean vector (computed strictly from X_train):
  age                      : 49.578
  sex                      : 0.325
  on thyroxine             : 0.052
  on antithyroid medication: 0.004
  pregnant                 : 0.003
  thyroid surgery          : 0.009
  lithium                  : 0.003
  goitre                   : 0.002
  tumor                    : 0.019
  TSH                      : 27.985
  T3                       : 1.686
  TT4                      : 67.144
  T4U                      : 0.995
  FTI                      : 81.237

StandardScaler artifact saved to ..\models\scaler.pkl


## 6. Verification of Scaled Data
After standardization, training features must have mean $\approx 0$ and standard deviation $\approx 1$.


In [6]:
df_scaled_train = pd.DataFrame(X_train_scaled, columns=X_train.columns)
print("Training Scaled Means (should be ~0.00):")
print(df_scaled_train.mean().round(3))
print("\nTraining Scaled Stds (should be ~1.00):")
print(df_scaled_train.std().round(3))


Training Scaled Means (should be ~0.00):
age                         -0.0
sex                         -0.0
on thyroxine                -0.0
on antithyroid medication    0.0
pregnant                     0.0
thyroid surgery             -0.0
lithium                      0.0
goitre                      -0.0
tumor                        0.0
TSH                          0.0
T3                           0.0
TT4                         -0.0
T4U                         -0.0
FTI                          0.0
dtype: float64

Training Scaled Stds (should be ~1.00):
age                          1.0
sex                          1.0
on thyroxine                 1.0
on antithyroid medication    1.0
pregnant                     1.0
thyroid surgery              1.0
lithium                      1.0
goitre                       1.0
tumor                        1.0
TSH                          1.0
T3                           1.0
TT4                          1.0
T4U                          1.0
FTI         